# Fixed-Length Speculative Decoding Baseline Sweep

This notebook sweeps fixed draft lengths from 1 to 8 across different workloads to establish the optimal static draft length baseline (the bar the adaptive controllers must beat).

Under **smoke mode**, we simulate the speculative decoding process using statistical acceptance profiles modeled after real-world benchmarks (Spec-Bench, MT-Bench, HumanEval, GSM8K).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Headless backend
import matplotlib.pyplot as plt
import os

# Set seeds for reproducibility
np.random.seed(42)
os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
print("Libraries imported successfully.")

## 1. Simulation Setup
We model the drafting and verification latencies:
- Target forward pass latency ($T_{target}$): **25.0ms**
- Draft forward pass latency ($T_{draft}$): **5.0ms**

The acceptance profiles reflect typical behaviors:
- **HumanEval (Code):** Long predictable spans. High probability of long acceptance sequences.
- **GSM8K (Math):** Step-by-step reasoning. Mixed predictability.
- **MT-Bench (Chat):** High variance. Boilerplate is easy, reasoning is hard.
- **Spec-Bench (Mixed):** Mixed overall traffic.

In [ ]:
T_TARGET = 25.0  # ms
T_DRAFT = 5.0    # ms

# Accept probabilities per index (1 to 8)
ACCEPT_PROBS = {
    "humaneval": [0.95, 0.92, 0.88, 0.85, 0.80, 0.75, 0.70, 0.65],
    "gsm8k":     [0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25, 0.15],
    "mt_bench":  [0.80, 0.70, 0.55, 0.45, 0.35, 0.20, 0.10, 0.05],
    "spec_bench":[0.85, 0.78, 0.70, 0.62, 0.52, 0.42, 0.32, 0.22]
}

def simulate_step(workload, K):
    """
    Simulates a single speculative decoding step.
    Returns (accepted_count, step_latency, ar_equivalent_latency)
    """
    probs = ACCEPT_PROBS[workload]
    accepted = 0
    for i in range(K):
        # Sample acceptance based on probability for this position
        if np.random.rand() < probs[i]:
            accepted += 1
        else:
            break
            
    # Speculative latency: K draft steps + 1 target pass
    spec_time = K * T_DRAFT + T_TARGET
    
    # AR equivalent: target passes for (accepted + 1) tokens
    ar_time = (accepted + 1) * T_TARGET
    
    return accepted, spec_time, ar_time

## 2. Running the Sweep
We run a simulation of 1000 generation steps for each fixed draft length $K \in \{1, 2, 3, 4, 5, 6, 7, 8\}$ on all 4 workloads.

In [ ]:
STEPS = 1000
workloads = ["humaneval", "gsm8k", "mt_bench", "spec_bench"]
draft_lengths = [1, 2, 3, 4, 5, 6, 7, 8]

results = []

for w in workloads:
    for K in draft_lengths:
        total_accepted = 0
        total_spec_time = 0.0
        total_ar_time = 0.0
        wasted_tokens = 0
        
        for _ in range(STEPS):
            acc, spec_t, ar_t = simulate_step(w, K)
            total_accepted += acc
            total_spec_time += spec_t
            total_ar_time += ar_t
            wasted_tokens += (K - acc)
            
        mean_acc_len = total_accepted / STEPS
        speedup = total_ar_time / total_spec_time
        wasted_per_accepted = wasted_tokens / (total_accepted + 1e-9)
        
        results.append({
            "workload": w,
            "draft_length": K,
            "mean_accepted_length": round(mean_acc_len, 3),
            "net_speedup": round(speedup, 3),
            "wasted_tokens_per_accepted": round(wasted_per_accepted, 3)
        })

df = pd.DataFrame(results)
df.to_csv("results/metrics_all.csv", index=False)
print("Sweep completed. Stored results in results/metrics_all.csv.")
df.head(16)

## 3. Finding the Best Static Draft Length per Workload

In [ ]:
print("=== Best Static Draft Length per Workload ===")
best_rows = []
for w in workloads:
    w_df = df[df["workload"] == w]
    best_idx = w_df["net_speedup"].idxmax()
    best_row = w_df.loc[best_idx]
    best_rows.append(best_row)
    print(f"Workload: {w:<12} | Best K: {best_row['draft_length']} | Speedup: {best_row['net_speedup']:.3f}x")

pd.DataFrame(best_rows)

## 4. Visualizing the Speedup Curves

In [ ]:
plt.figure(figsize=(10, 6))
for w in workloads:
    w_df = df[df["workload"] == w]
    plt.plot(w_df["draft_length"], w_df["net_speedup"], marker='o', label=w)

plt.title("Speculative Decoding Speedup vs. Fixed Draft Length (K)")
plt.xlabel("Draft Length (K)")
plt.ylabel("Net Speedup (x)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.savefig("results/figures/fig1_tradeoff.png", dpi=300)
plt.show()